In [0]:
# Test xem Spark có hoạt động không
print(spark.version)
print("Spark đang chạy")

4.1.0
Spark đang chạy


In [0]:
# ============================================================
#  01_BRONZE_INGESTION — TOURGO DATA PIPELINE
#  Mục đích: Đọc 6 CSV từ S3 raw/ → lưu vào Bronze Delta Managed Tables

import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

import boto3
import pandas as pd
import io

AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BUCKET_NAME    = ""
REGION         = "eu-north-1"

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

# Tạo một Database riêng tên là 'tourgo' để lưu trữ các bảng
spark.sql("CREATE DATABASE IF NOT EXISTS tourgo")
spark.sql("USE tourgo")

def read_csv_from_s3(table_name):
    key = f"raw/{table_name}/{table_name}.csv"
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    content = obj['Body'].read().decode('utf-8-sig')
    pandas_df = pd.read_csv(io.StringIO(content))
    spark_df  = spark.createDataFrame(pandas_df)
    return spark_df

print("-> Kết nối S3 sẵn sàng & Sử dụng Database: tourgo")


-> Kết nối S3 sẵn sàng & Sử dụng Database: tourgo


In [0]:
# ============================================================
#  BRONZE LAYER: Lưu nguyên bản từ S3 → Delta Managed Table (tourgo)


TABLES = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']

bronze_summary = {}

for table in TABLES:
    print(f"\n{'='*50}")
    print(f"ĐANG INGEST: {table.upper()}")

    try:
        df = read_csv_from_s3(table)
        count = df.count()

        # Lưu dưới dạng Delta Managed Table
        table_name = f"bronze_{table}"
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)

        bronze_summary[table] = {"status": "--OK--", "count": count, "columns": df.columns}
        print(f"   --OK-- Lưu xong bảng: {table_name} — {count} records | Columns: {list(df.columns)}")

    except Exception as e:
        bronze_summary[table] = {"status": "--LỖI--", "error": str(e)}
        print(f"--LỖI--: {e}")



ĐANG INGEST: USERS
   --OK-- Lưu xong bảng: bronze_users — 18 records | Columns: ['id', 'role', 'is_active', 'date_joined']

ĐANG INGEST: TOURS
   --OK-- Lưu xong bảng: bronze_tours — 23 records | Columns: ['id', 'creator_id', 'title', 'price', 'departure_date', 'slots', 'status', 'created_at', 'category_names']

ĐANG INGEST: BOOKINGS
   --OK-- Lưu xong bảng: bronze_bookings — 11 records | Columns: ['id', 'user_id', 'tour_id', 'number_of_people', 'total_price', 'booking_date', 'status', 'created_at']

ĐANG INGEST: PAYMENTS
   --OK-- Lưu xong bảng: bronze_payments — 10 records | Columns: ['id', 'booking_id', 'amount', 'payment_method', 'status', 'created_at']

ĐANG INGEST: REVENUES
   --OK-- Lưu xong bảng: bronze_revenues — 4 records | Columns: ['id', 'payment_id', 'creator_id', 'total_amount', 'creator_share', 'admin_share', 'created_at']

ĐANG INGEST: REVIEWS
   --OK-- Lưu xong bảng: bronze_reviews — 1 records | Columns: ['id', 'user_id', 'tour_id', 'rating', 'created_at']


In [0]:
# ============================================================
#  BRONZE SUMMARY — In ra để chụp screenshot

print("\n" + "="*55)
print("BRONZE INGESTION SUMMARY")
print("="*55)
print(f"{'Table':<12} | {'Status':<8} | {'Records':>8}")
print("-"*35)

total_ok = 0
for t, r in bronze_summary.items():
    if r["status"] == "--OK--":
        print(f"{t:<12} | {'OK':<8} | {r['count']:>8,}")
        total_ok += 1
    else:
        print(f"{t:<12} | {'LỖI':<8} | {'N/A':>8}  ← {r.get('error','')[:40]}")

print("-"*35)
print(f"{'TỔNG':<12} | {total_ok}/6 OK")
print("="*55)

# Verify đọc lại từ Delta để chắc chắn đã lưu được
print("\n VERIFY — Đọc lại từ Bronze Delta Tables:")
for t in TABLES:
    try:
        table_name = f"bronze_{t}"
        count = spark.read.table(table_name).count()
        print(f"   Bảng {table_name:<15} → {count:,} records --OK--")
    except Exception as e:
        print(f"   Bảng bronze_{t:<12} → --LỖI-- {e}")



📊 BRONZE INGESTION SUMMARY
Table        | Status   |  Records
-----------------------------------
users        | OK       |       18
tours        | OK       |       23
bookings     | OK       |       11
payments     | OK       |       10
revenues     | OK       |        4
reviews      | OK       |        1
-----------------------------------
TỔNG         | 6/6 OK

 VERIFY — Đọc lại từ Bronze Delta Tables:
   Bảng bronze_users    → 18 records --OK--
   Bảng bronze_tours    → 23 records --OK--
   Bảng bronze_bookings → 11 records --OK--
   Bảng bronze_payments → 10 records --OK--
   Bảng bronze_revenues → 4 records --OK--
   Bảng bronze_reviews  → 1 records --OK--


In [0]:
# ============================================================
#  PREVIEW — Xem 2 dòng đầu từng Bronze table
# ============================================================

for t in TABLES:
    table_name = f"bronze_{t}"
    print(f"\n{'='*50}")
    print(f" BRONZE PREVIEW: {table_name.upper()}")
    try:
        df = spark.read.table(table_name)
        df.printSchema()
        df.show(2, truncate=False)
    except Exception as e:
        print(f"--LỖI-- Không đọc được: {e}")



 BRONZE PREVIEW: BRONZE_USERS
root
 |-- id: long (nullable = true)
 |-- role: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- date_joined: string (nullable = true)

+---+--------+---------+-------------------+
|id |role    |is_active|date_joined        |
+---+--------+---------+-------------------+
|6  |Admin   |true     |2026-05-22 05:30:47|
|10 |Customer|true     |2026-05-23 10:28:57|
+---+--------+---------+-------------------+
only showing top 2 rows

 BRONZE PREVIEW: BRONZE_TOURS
root
 |-- id: long (nullable = true)
 |-- creator_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price: double (nullable = true)
 |-- departure_date: string (nullable = true)
 |-- slots: long (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- category_names: string (nullable = true)

+---+----------+------------------------------------+--------+--------------+-----+--------+-------------------+--------------